# Hausa Medical ASR — Full Evaluation Notebook

Evaluates three decoding strategies on the human-recorded evaluation set:

1. **Greedy** — no language model
2. **Shallow Fusion (SF)** — CTC + medical language model
3. **Density Ratio Approach (DRA)** — CTC + medical LM − general LM

**Metrics:** WER and CER per utterance, corpus-level (micro-averaged) WER/CER,
per-tier / per-sex / per-speaker breakdowns, bootstrap confidence intervals,
and matched-pairs significance tests.

**Outputs:** `hausa-asr/6_results/evaluation_results_2.csv` (per-utterance),
`summary_results_2.csv` (aggregated), `sentence_results_2.csv`, `mter_results.csv`.


## 1. Setup & Installation

In [1]:
print("Clean install")
import subprocess

# Uninstall first
for pkg in ["numpy", "pandas", "numba", "pyctcdecode", "librosa", "jiwer", "transformers", "scipy"]:
    subprocess.run(f"pip uninstall -y {pkg}", shell=True, capture_output=True)

# Install numpy FIRST and ALONE, verify it works before anything else touches it
result = subprocess.run("pip install -q numpy==1.26.4", shell=True, capture_output=True, text=True)
print("numpy install:", "OK" if result.returncode == 0 else result.stderr[:300])

Clean install
numpy install: OK


In [3]:
# Install pinned, mutually compatible package versions in a single pass.
import subprocess

PACKAGES_TO_PIN = [
    "numpy", "pandas", "numba", "pyctcdecode",
    "librosa", "jiwer", "transformers", "scipy",
]

for pkg in PACKAGES_TO_PIN:
    subprocess.run(f"pip uninstall -y {pkg}", shell=True, capture_output=True)

install_cmds = [
    "pip install -q numpy==1.26.4",
    "pip install -q pandas==2.2.2",
    "pip install -q numba==0.59.1",
    "pip install -q pyctcdecode==0.5.0",
    "pip install -q librosa",
    "pip install -q jiwer",
    "pip install -q scipy",
    "pip install -q transformers",
    "pip install -q https://github.com/kpu/kenlm/archive/master.zip",
    "apt-get install -y ffmpeg -q",
]

for cmd in install_cmds:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    status = "OK" if result.returncode == 0 else f"FAILED: {result.stderr[:200]}"
    print(f"{cmd:<55} {status}")

import transformers
print(f"\nDependencies ready. transformers=={transformers.__version__}")


pip install -q numpy==1.26.4                            OK
pip install -q pandas==2.2.2                            OK
pip install -q numba==0.59.1                            OK
pip install -q pyctcdecode==0.5.0                       OK
pip install -q librosa                                  OK
pip install -q jiwer                                    OK
pip install -q scipy                                    OK
pip install -q transformers                             OK
pip install -q https://github.com/kpu/kenlm/archive/master.zip OK
apt-get install -y ffmpeg -q                            OK

Dependencies ready. transformers==5.17.0


## 2. Imports & Configuration

In [4]:
import os
import re
import csv
import json
import time
import shutil
import subprocess
import numpy as np
import pandas as pd
import torch
import librosa
import kenlm
from pathlib import Path
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
import scipy
print(f"SciPy version: {scipy.__version__}")
print("Imports successful. Internal SciPy linkage is restored.")

NumPy version: 2.5.3
Pandas version: 2.2.2
SciPy version: 1.18.1
Imports successful. Internal SciPy linkage is restored.


In [5]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [6]:
BASE_DIR    = "/content/drive/MyDrive/hausa-asr"
EVAL_DIR    = f"{BASE_DIR}/5_evaluation_dataset"
AUDIO_DIR   = f"{EVAL_DIR}/audio"
METADATA    = f"{EVAL_DIR}/metadata.csv"
LM_TGT      = f"{BASE_DIR}/3_language_models/hausa_health_4gram.bin"
LM_SRC      = f"{BASE_DIR}/3_language_models/hausa_general_4gram.bin"
RESULTS_DIR = f"{BASE_DIR}/6_results"
MODEL_NAME  = "facebook/mms-1b-fl102"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("/content/wav_cache", exist_ok=True)  # local wav conversion cache

print(f"Device      : {DEVICE}")
print(f"Eval dir    : {EVAL_DIR}")
print(f"Results dir : {RESULTS_DIR}")


Device      : cuda
Eval dir    : /content/drive/MyDrive/hausa-asr/5_evaluation_dataset
Results dir : /content/drive/MyDrive/hausa-asr/6_results


## 3. Text Normalization

In [7]:
def normalize_text(text: str) -> str:
    """Lowercase + strip punctuation, keeping Hausa special characters."""
    text = text.lower()
    text = re.sub(r"[^\w\s\u0253\u0257\u0199\u01b4\u02bc]", "", text)
    return " ".join(text.split())


## 4. Load Evaluation Metadata

In [8]:
print("Loading evaluation metadata")

if not os.path.exists(METADATA):
    raise FileNotFoundError(
        f"metadata.csv not found at {METADATA}\n"
        f"Please place metadata.csv in: {EVAL_DIR}"
    )

df_meta = pd.read_csv(METADATA, encoding="utf-8")

# Verify audio files exist and report missing
df_meta["full_audio_path"] = df_meta["audio_path"].apply(
    lambda p: os.path.join(EVAL_DIR, p)
)
df_meta["file_exists"] = df_meta["full_audio_path"].apply(os.path.exists)

total   = len(df_meta)
found   = df_meta["file_exists"].sum()
missing = total - found

print(f"  Total utterances expected : {total}")
print(f"  Audio files found         : {found}")
print(f"  Audio files missing       : {missing}")

if missing > 0:
    print("\n  Missing files:")
    for _, row in df_meta[~df_meta["file_exists"]].iterrows():
        print(f"    {row['full_audio_path']}")

# Work only with found files
df_eval = df_meta[df_meta["file_exists"]].reset_index(drop=True)
print(f"\n  Proceeding with {len(df_eval)} utterances from "
      f"{df_eval['speaker_id'].nunique()} speakers")

print("\n  Speaker breakdown:")
for spk in sorted(df_eval["speaker_id"].unique()):
    subset = df_eval[df_eval["speaker_id"] == spk].iloc[0]
    count  = (df_eval["speaker_id"] == spk).sum()
    print(f"    {spk} | {subset['state']:<10} | {subset['sex']} | {count} files")


Loading evaluation metadata
  Total utterances expected : 250
  Audio files found         : 250
  Audio files missing       : 0

  Proceeding with 250 utterances from 10 speakers

  Speaker breakdown:
    S04 | Plateau    | M | 25 files
    S06 | Gombe      | M | 25 files
    S08 | Taraba     | M | 25 files
    S10 | Yobe       | M | 25 files
    S11 | Kaduna     | M | 25 files
    S13 | Gombe      | F | 25 files
    S18 | Gombe      | F | 25 files
    S20 | Bauchi     | F | 25 files
    S21 | Bauchi     | F | 25 files
    S23 | Gombe      | F | 25 files


## 5. N-gram Overlap Analysis (Eval Set vs. Medical LM Corpus)

Checks how much of the evaluation set's phrasing already appears verbatim in
the medical LM's training corpus -- if overlap is high, part of the medical
LM's advantage could be phrase memorization rather than genuine domain
adaptation, which matters for how the WER/MTER gains should be interpreted.

**Update `MEDICAL_CORPUS_PATH` below if your raw medical/health corpus text
file has a different name** -- this notebook only has the trained `.bin` LM
path (`LM_TGT`), not the raw training text, so the path is a best guess
based on your naming convention for the binary model.

In [9]:
import os

print("MEDICAL CORPUS CHECK")

# Search the folders where your medical corpus files appear to live
search_dirs = [
    f"{BASE_DIR}/2_processed_text",
    f"{BASE_DIR}/3_language_models",
    f"{BASE_DIR}/4_data",
    BASE_DIR
]

found_files = []

for directory in search_dirs:
    if not os.path.exists(directory):
        continue

    for filename in os.listdir(directory):
        if filename.lower().endswith((".txt", ".text")):
            if any(term in filename.lower() for term in ["health", "medical", "corpus"]):
                path = os.path.join(directory, filename)

                if path not in found_files:
                    found_files.append(path)

print(f"\nFound {len(found_files)} candidate medical corpus files:\n")

for path in found_files:
    try:
        with open(path, encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        print("-" * 80)
        print(f"FILE: {path}")
        print(f"Sentences/lines: {len(lines):,}")

        # Basic word count
        word_count = sum(len(line.split()) for line in lines)
        print(f"Words: {word_count:,}")

    except Exception as e:
        print(f"ERROR reading {path}: {e}")

print("TARGET")
print("""
paper reports:
    Final medical corpus: 19,421 sentences
    ~590,000 words

identify the EXACT file matching that corpus.
before running the n-gram overlap analysis until we confirm this.
""")

MEDICAL CORPUS CHECK

Found 6 candidate medical corpus files:

--------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/hausa-asr/2_processed_text/hausa_health_clean.txt
Sentences/lines: 19,421
Words: 558,737
--------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/hausa-asr/2_processed_text/hausa_health_train.txt
Sentences/lines: 17,478
Words: 502,925
--------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/hausa-asr/2_processed_text/hausa_health_dev.txt
Sentences/lines: 1,943
Words: 55,812
--------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/hausa-asr/2_processed_text/hausa_health_train_cleaned.txt
Sentences/lines: 17,477
Words: 449,442
--------------------------------------------------------------------------------
FILE: /content/drive/MyDrive/hausa-asr/2_processed_text/

In [10]:
N_GRAM_N = 4

# TODO: confirm this matches your actual corpus text file name/location.
MEDICAL_CORPUS_PATH = f"{BASE_DIR}/2_processed_text/hausa_health_clean.txt"

with open(MEDICAL_CORPUS_PATH, encoding="utf-8") as f:
    corpus_lines = [line.strip() for line in f if line.strip()]

print(f"Medical LM training corpus: {len(corpus_lines)} lines")


def get_ngrams(text: str, n: int) -> set:
    words = normalize_text(text).split()
    if len(words) < n:
        return set()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


corpus_ngrams = set()
for line in corpus_lines:
    corpus_ngrams |= get_ngrams(line, N_GRAM_N)

print(f"Unique {N_GRAM_N}-grams in training corpus: {len(corpus_ngrams):,}")

overlap_rows = []
for _, row in df_eval.iterrows():
    eval_ngrams = get_ngrams(str(row["hausa_reference"]), N_GRAM_N)
    matched     = eval_ngrams & corpus_ngrams
    overlap_rows.append({
        "sentence_id": row["sentence_id"],
        "reference"  : str(row["hausa_reference"])[:50],
        "n_ngrams"   : len(eval_ngrams),
        "n_matched"  : len(matched),
        "pct_matched": round(len(matched) / len(eval_ngrams) * 100, 1) if eval_ngrams else 0.0,
    })

df_overlap = pd.DataFrame(overlap_rows).drop_duplicates(subset="sentence_id")

print(f"\n{'Sent':<6} {'% overlap':>9}  Reference")
for _, r in df_overlap.iterrows():
    print(f"{r['sentence_id']:<6} {r['pct_matched']:>8.1f}%  {r['reference']}...")

n_sentences        = len(df_overlap)
n_any_overlap      = int((df_overlap["n_matched"] > 0).sum())
total_eval_ngrams  = int(df_overlap["n_ngrams"].sum())
total_matched      = int(df_overlap["n_matched"].sum())
pooled_pct_overlap = round(total_matched / total_eval_ngrams * 100, 1) if total_eval_ngrams else 0.0

print(f"""
N-gram overlap summary ({N_GRAM_N}-grams)
  Evaluation sentences                    : {n_sentences}
  Sentences with >=1 overlapping {N_GRAM_N}-gram    : {n_any_overlap} ({n_any_overlap/n_sentences*100:.1f}%)
  Total {N_GRAM_N}-grams across all sentences (pooled) : {total_eval_ngrams}
  Matched against training corpus         : {total_matched} ({pooled_pct_overlap}%)
""")

report_sentence = (
    f"To assess potential overlap between the evaluation set and the medical "
    f"LM corpus, we measured {N_GRAM_N}-gram overlap between the "
    f"{n_sentences} evaluation sentences and the LM training corpus "
    f"({len(corpus_lines)} lines). {n_any_overlap} of {n_sentences} evaluation "
    f"sentences ({n_any_overlap/n_sentences*100:.1f}%) shared at least one "
    f"{N_GRAM_N}-gram with the training corpus, and {pooled_pct_overlap}% of "
    f"all evaluation {N_GRAM_N}-grams (pooled) appeared in the training data."
)
print(report_sentence)

overlap_path = f"{RESULTS_DIR}/ngram_overlap_analysis.csv"
df_overlap.to_csv(overlap_path, index=False, encoding="utf-8")
print(f"\nSaved: {overlap_path}")


Medical LM training corpus: 19421 lines
Unique 4-grams in training corpus: 422,112

Sent   % overlap  Reference
1          25.0%  Ka sha ruwa mai yawa kowace rana....
2           0.0%  Ina jin zafi a kirjina....
3           0.0%  Jikinna ya kama zafi yau sosai....
4           0.0%  Ina buƙatar magani don ciwon kai....
5           0.0%  Ka huta sosai don ka warke da wuri....
6           0.0%  Ina jin kishin ruwa sosai....
7           0.0%  Hannuwana na karkarwa....
8           0.0%  Ina tai amai sau da yawa yau....
9          11.1%  Likita ya ce ina da zazzabin cizon sauro kuma ya r...
10         12.5%  Ka sha ƙwayar maganin sau uku a kowace rana bayan ...
11          0.0%  An diba jinina a asibiti domin gwaji....
12         12.5%  Ciwon sukari yana buƙatar kulawa ta musamman da ab...
13          0.0%  Ina da tari mai tsanani tun sati ɗaya da ya wuce....
14          0.0%  Likita ya umurce ni da in dawo bayan kwana bakwai ...
15          0.0%  Yaron yana da gudawa kuma jikansa ya yi raun

## 6. Audio Conversion

In [11]:
def convert_to_wav(audio_path: str, cache_dir: str = "/content/wav_cache") -> str:
    """
    Convert any audio format to 16kHz mono WAV using ffmpeg.
    Caches converted files to avoid repeated conversion.
    Returns path to the converted WAV file.
    """
    stem     = Path(audio_path).stem
    wav_path = os.path.join(cache_dir, f"{stem}.wav")

    if os.path.exists(wav_path):
        return wav_path  # already converted

    cmd = (
        f"ffmpeg -i '{audio_path}' "
        f"-ar 16000 -ac 1 -acodec pcm_s16le "
        f"'{wav_path}' -y -loglevel error"
    )
    result = subprocess.run(cmd, shell=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"ffmpeg conversion failed for {audio_path}:\n"
            f"{result.stderr.decode()}"
        )
    return wav_path


## 7. Load ASR Model

In [12]:
print("Loading MMS-1B-FL102 (Hausa adapter)")

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME, target_lang="hau")
model     = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

model.load_adapter("hau")
model.to(DEVICE).eval()

print(f"  Device  : {DEVICE}")
print(f"  Adapter : hau (Hausa)")

# Vocabulary for pyctcdecode
vocab_dict = processor.tokenizer.get_vocab()
vocab_list = [tok for tok, _ in sorted(vocab_dict.items(), key=lambda x: x[1])]
print(f"  Vocabulary size: {len(vocab_list)} tokens")


Loading MMS-1B-FL102 (Hausa adapter)


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/351k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.hau.safetensors: reconstructing file:   0%|          |  0.00B / 9.04MB            

adapter.hau.safetensors: downloading bytes:           |  0.00B            

  Device  : cuda
  Adapter : hau (Hausa)
  Vocabulary size: 79 tokens


## 8. Build Decoders

In [13]:
print("Building decoders")

# Shallow Fusion decoder
has_tgt    = os.path.exists(LM_TGT)
decoder_sf = None

if has_tgt:
    print(f"  Target LM : {LM_TGT} ({os.path.getsize(LM_TGT)/1e6:.1f} MB)")
    decoder_sf = build_ctcdecoder(
        labels=vocab_list,
        kenlm_model_path=LM_TGT,
        alpha=0.5,
        beta=1.0,
    )
    print("  Shallow Fusion decoder ready.")
else:
    print(f"  WARNING: Target LM not found at {LM_TGT}")

# KenLM models for DRA rescoring
has_src   = os.path.exists(LM_SRC)
kenlm_tgt = kenlm.Model(LM_TGT) if has_tgt else None
kenlm_src = kenlm.Model(LM_SRC) if has_src else None

if has_src:
    print(f"  Source LM : {LM_SRC} ({os.path.getsize(LM_SRC)/1e6:.1f} MB)")
    print("  DRA fully active.")
else:
    print(f"  WARNING: Source LM not found — DRA falls back to Shallow Fusion (lambda_psi=0)")


Building decoders
  Target LM : /content/drive/MyDrive/hausa-asr/3_language_models/hausa_health_4gram.bin (17.6 MB)


  Shallow Fusion decoder ready.
  Source LM : /content/drive/MyDrive/hausa-asr/3_language_models/hausa_general_4gram.bin (39.0 MB)
  DRA fully active.


## 9. Inference Functions

In [14]:
def get_logits(wav_path: str) -> np.ndarray:
    """Load wav and return CTC logits [T, vocab]."""
    audio, _ = librosa.load(wav_path, sr=16000)
    inputs   = processor(audio, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(inputs.input_values).logits
    return logits[0].cpu().numpy()


def transcribe_greedy(logits: np.ndarray) -> str:
    pred_ids = np.argmax(logits, axis=-1)
    return processor.decode(pred_ids)


def set_sf_lm_params(decoder, alpha: float, beta: float) -> None:
    """
    Update the KenLM interpolation weights on an already-built pyctcdecode
    decoder without reloading the KenLM model from disk. This is what makes
    the hyperparameter sweep in Section 11 fast: swap alpha/beta, re-decode
    the cached logits, no rebuild needed.
    """
    decoder._language_model.alpha = alpha
    decoder._language_model.beta  = beta


def transcribe_sf(logits: np.ndarray,
                   lm_weight: float = 0.5,
                   word_score: float = 1.0,
                   beam_width: int = 100) -> str:
    if decoder_sf is None:
        return transcribe_greedy(logits)
    set_sf_lm_params(decoder_sf, lm_weight, word_score)
    return decoder_sf.decode(
        logits,
        beam_width=beam_width,
        beam_prune_logp=-10.0,
        token_min_logp=-5.0,
    )


LN10 = 2.302585  # convert log10 (KenLM) to natural log


def transcribe_dra(logits: np.ndarray,
                    lambda_tau: float = 0.8,
                    lambda_psi: float = 0.5,
                    word_score: float = 1.0,
                    beam_width: int = 100,
                    nbest: int = 50) -> str:
    """
    Density Ratio Approach:
        score = CTC + lambda_tau * log P_TGT - lambda_psi * log P_SRC
    """
    if decoder_sf is None:
        return transcribe_greedy(logits)

    beams = decoder_sf.decode_beams(
        logits,
        beam_width=beam_width,
        beam_prune_logp=-10.0,
        token_min_logp=-5.0,
    )[:nbest]

    if not beams:
        return transcribe_greedy(logits)

    best_text  = beams[0][0]
    best_score = -1e18

    for beam in beams:
        hyp_text    = beam[0]
        ctc_logprob = beam[3]
        lp_tgt = (kenlm_tgt.score(hyp_text, bos=True, eos=True) * LN10
                  if kenlm_tgt else 0.0)
        lp_src = (kenlm_src.score(hyp_text, bos=True, eos=True) * LN10
                  if kenlm_src else 0.0)
        n_words  = max(len(hyp_text.split()), 1)
        dr_score = (ctc_logprob
                    + lambda_tau * lp_tgt
                    - lambda_psi * lp_src
                    + word_score * n_words)
        if dr_score > best_score:
            best_score = dr_score
            best_text  = hyp_text

    return best_text


## 10. Cache CTC Logits Once

Runs the GPU forward pass once for every evaluation utterance and stores the raw CTC logits. All cross-validation tuning, held-out decoding, diagnostics, and ablations reuse this cache; no hyperparameter configuration reruns the acoustic model.

In [15]:
print(f"Caching CTC logits for {len(df_eval)} evaluation utterances")
print("One GPU forward pass per utterance; all later decoding reuses this cache.")

logits_cache = {}
refs_cache   = {}

for i, row in df_eval.iterrows():
    wav_path = convert_to_wav(row["full_audio_path"])
    logits_cache[row["file_id"]] = get_logits(wav_path)
    refs_cache[row["file_id"]]   = normalize_text(str(row["hausa_reference"]))
    if (i + 1) % 10 == 0 or (i + 1) == len(df_eval):
        print(f"  Cached {i + 1}/{len(df_eval)}")

print(f"\nDone -- {len(logits_cache)} utterances cached.")

Caching CTC logits for 250 evaluation utterances
One GPU forward pass per utterance; all later decoding reuses this cache.
  Cached 10/250
  Cached 20/250
  Cached 30/250
  Cached 40/250
  Cached 50/250
  Cached 60/250
  Cached 70/250
  Cached 80/250
  Cached 90/250
  Cached 100/250
  Cached 110/250
  Cached 120/250
  Cached 130/250
  Cached 140/250
  Cached 150/250
  Cached 160/250
  Cached 170/250
  Cached 180/250
  Cached 190/250
  Cached 200/250
  Cached 210/250
  Cached 220/250
  Cached 230/250
  Cached 240/250
  Cached 250/250

Done -- 250 utterances cached.


## 11. Sentence-Level 5-Fold Cross-Validation

**ICASSP evaluation protocol.** The 25 unique sentence prompts are partitioned into five deterministic folds of five prompts each. In every fold, all 10 speaker recordings are retained: 20 prompts (200 utterances) are used **only for decoding-parameter selection**, while the remaining 5 prompts (50 utterances) are held out for final evaluation.

The MMS-1B-FL102 acoustic model is fixed throughout this experiment; no acoustic-model selection is performed on the evaluation folds. SF, General-LM SF, and DRA parameters are selected independently inside each training/tuning fold and then frozen before decoding the five unseen prompts. The five held-out sets are pooled to obtain 250 out-of-fold predictions.

In [16]:
# -----------------------------------------------------------------------------
# Fixed ICASSP evaluation protocol
# -----------------------------------------------------------------------------
from jiwer import wer as compute_wer, cer as compute_cer
import itertools

N_FOLDS = 5
RANDOM_SEED = 42
BEAM_WIDTH = 100
DRA_NBEST = 50

LM_WEIGHT_GRID = [0.2, 0.35, 0.5, 0.65, 0.8, 1.0]
WORD_SCORE_GRID = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
LAMBDA_TAU_GRID = [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]
LAMBDA_PSI_GRID = [0.1, 0.3, 0.5, 0.8, 1.0]
DRA_WORD_SCORE_GRID = [0.5, 1.0, 1.5]

# Normalize IDs so the grouping is robust to CSV numeric/string formatting.
df_eval["sentence_id_norm"] = df_eval["sentence_id"].astype(str).str.zfill(3)

unique_sentences = sorted(df_eval["sentence_id_norm"].unique())
print(f"Unique sentence prompts: {len(unique_sentences)}")
print(f"Unique speakers        : {df_eval['speaker_id'].nunique()}")
print(f"Total utterances       : {len(df_eval)}")

assert len(unique_sentences) == 25, (
    f"Expected exactly 25 unique sentence prompts, found {len(unique_sentences)}"
)
counts_per_sentence = df_eval.groupby("sentence_id_norm").size()
assert counts_per_sentence.eq(10).all(), (
    "Every sentence must have exactly 10 speaker recordings for the planned "
    "5-fold sentence-cluster evaluation."
)

# Deterministic five-fold split. Each fold has five unique sentence prompts.
rng = np.random.default_rng(RANDOM_SEED)
shuffled_sentences = unique_sentences.copy()
rng.shuffle(shuffled_sentences)
fold_sentence_ids = [sorted(x.tolist()) for x in np.array_split(shuffled_sentences, N_FOLDS)]

for k, fold_sents in enumerate(fold_sentence_ids, start=1):
    print(f"Fold {k}: {fold_sents} | tune={len(fold_sents)*0 + 20} prompts, held-out=5 prompts")

assert sorted(sum(fold_sentence_ids, [])) == unique_sentences


Unique sentence prompts: 25
Unique speakers        : 10
Total utterances       : 250
Fold 1: ['010', '016', '017', '020', '021'] | tune=20 prompts, held-out=5 prompts
Fold 2: ['007', '008', '011', '018', '025'] | tune=20 prompts, held-out=5 prompts
Fold 3: ['001', '004', '013', '019', '022'] | tune=20 prompts, held-out=5 prompts
Fold 4: ['003', '006', '012', '015', '024'] | tune=20 prompts, held-out=5 prompts
Fold 5: ['002', '005', '009', '014', '023'] | tune=20 prompts, held-out=5 prompts


### Fold-level decoding helpers

The helpers below make the separation explicit: parameter search sees only the tuning prompts, while final held-out decoding sees only frozen fold parameters. DRA still uses the top-50 candidates generated by the SF beam, but the SF beam itself is configured using the **fold's SF parameters**, never parameters selected from the held-out prompts.

In [17]:
def decode_sf_with_params(logits, alpha, beta, decoder=None):
    """Decode one utterance with explicitly supplied SF parameters."""
    dec = decoder if decoder is not None else decoder_sf
    if dec is None:
        return transcribe_greedy(logits)
    set_sf_lm_params(dec, float(alpha), float(beta))
    return dec.decode(
        logits, beam_width=BEAM_WIDTH,
        beam_prune_logp=-10.0, token_min_logp=-5.0,
    )


def decode_dra_with_params(logits, lambda_tau, lambda_psi, word_score,
                           sf_alpha, sf_beta, decoder=None, nbest=DRA_NBEST):
    """DRA n-best rescoring over a fold-specific SF beam."""
    dec = decoder if decoder is not None else decoder_sf
    if dec is None:
        return transcribe_greedy(logits)

    set_sf_lm_params(dec, float(sf_alpha), float(sf_beta))
    beams = dec.decode_beams(
        logits, beam_width=BEAM_WIDTH,
        beam_prune_logp=-10.0, token_min_logp=-5.0,
    )[:nbest]

    if not beams:
        return transcribe_greedy(logits)

    best_text = beams[0][0]
    best_score = -1e18
    for beam in beams:
        hyp_text = beam[0]
        ctc_logprob = beam[3]
        lp_tgt = kenlm_tgt.score(hyp_text, bos=True, eos=True) * LN10 if kenlm_tgt else 0.0
        lp_src = kenlm_src.score(hyp_text, bos=True, eos=True) * LN10 if kenlm_src else 0.0
        n_words = max(len(hyp_text.split()), 1)
        score = (ctc_logprob + float(lambda_tau) * lp_tgt
                 - float(lambda_psi) * lp_src + float(word_score) * n_words)
        if score > best_score:
            best_score = score
            best_text = hyp_text
    return best_text


def tune_sf_on_ids(file_ids):
    """Select SF alpha/beta using only the supplied tuning utterances."""
    refs = [refs_cache[fid] for fid in file_ids]
    rows = []
    for alpha in LM_WEIGHT_GRID:
        for beta in WORD_SCORE_GRID:
            hyps = [decode_sf_with_params(logits_cache[fid], alpha, beta) for fid in file_ids]
            score = compute_wer(refs, hyps) * 100
            rows.append({"lm_weight": alpha, "word_score": beta, "wer": score})
    return pd.DataFrame(rows).sort_values(["wer", "lm_weight", "word_score"]).reset_index(drop=True)


def tune_general_sf_on_ids(file_ids, decoder_general):
    """Tune General-LM-only SF on the same tuning prompts, separately per fold."""
    refs = [refs_cache[fid] for fid in file_ids]
    rows = []
    for alpha in LM_WEIGHT_GRID:
        for beta in WORD_SCORE_GRID:
            hyps = [decode_sf_with_params(logits_cache[fid], alpha, beta, decoder=decoder_general)
                    for fid in file_ids]
            score = compute_wer(refs, hyps) * 100
            rows.append({"lm_weight": alpha, "word_score": beta, "wer": score})
    return pd.DataFrame(rows).sort_values(["wer", "lm_weight", "word_score"]).reset_index(drop=True)


def tune_dra_on_ids(file_ids, sf_alpha, sf_beta):
    """Select DRA parameters using only the supplied tuning utterances."""
    refs = [refs_cache[fid] for fid in file_ids]
    configs = [
        (t, p, w) for t, p, w in itertools.product(
            LAMBDA_TAU_GRID, LAMBDA_PSI_GRID, DRA_WORD_SCORE_GRID
        ) if p < t
    ]
    rows = []
    for lambda_tau, lambda_psi, word_score in configs:
        hyps = [
            decode_dra_with_params(
                logits_cache[fid], lambda_tau, lambda_psi, word_score,
                sf_alpha, sf_beta, decoder=decoder_sf, nbest=DRA_NBEST
            ) for fid in file_ids
        ]
        score = compute_wer(refs, hyps) * 100
        rows.append({
            "lambda_tau": lambda_tau, "lambda_psi": lambda_psi,
            "word_score": word_score, "wer": score
        })
    return pd.DataFrame(rows).sort_values(
        ["wer", "lambda_tau", "lambda_psi", "word_score"]
    ).reset_index(drop=True)


## 12. Cross-Validated Parameter Selection and Out-of-Fold Evaluation

For each fold we tune on 200 utterances (20 prompts × 10 speakers), freeze the selected parameters, and decode the 50 held-out utterances (5 prompts × 10 speakers). Greedy decoding is parameter-free. The final ICASSP predictions are the concatenation of the five held-out folds.

In [18]:
# Build a separate General-LM decoder for the properly tuned ablation.
decoder_general_only = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=LM_SRC,
    alpha=0.5, beta=1.0,
) if has_src else None

cv_records = []
fold_param_records = []
fold_sweep_records = []

for fold_idx, heldout_sents in enumerate(fold_sentence_ids, start=1):
    heldout_sents = set(heldout_sents)
    tune_df = df_eval[~df_eval["sentence_id_norm"].isin(heldout_sents)]
    test_df = df_eval[df_eval["sentence_id_norm"].isin(heldout_sents)]

    tune_ids = tune_df["file_id"].tolist()
    test_ids = test_df["file_id"].tolist()

    assert len(tune_df) == 200 and len(test_df) == 50

    print("\n" + "=" * 90)
    print(f"FOLD {fold_idx}/5")
    print(f"  Tuning prompts : {sorted(set(tune_df['sentence_id_norm']))}")
    print(f"  Held-out prompts: {sorted(heldout_sents)}")
    print(f"  Tuning utterances: {len(tune_ids)} | Held-out utterances: {len(test_ids)}")

    # 1) Tune medical-LM SF only on tuning prompts.
    sf_sweep = tune_sf_on_ids(tune_ids)
    best_sf_fold = sf_sweep.iloc[0]
    sf_alpha = float(best_sf_fold["lm_weight"])
    sf_beta = float(best_sf_fold["word_score"])

    sf_sweep.insert(0, "fold", fold_idx)
    sf_sweep["condition"] = "medical_sf"
    fold_sweep_records.append(sf_sweep)

    # 2) Tune General-LM-only SF independently on tuning prompts.
    if decoder_general_only is not None:
        gen_sweep = tune_general_sf_on_ids(tune_ids, decoder_general_only)
        best_gen_fold = gen_sweep.iloc[0]
        gen_alpha = float(best_gen_fold["lm_weight"])
        gen_beta = float(best_gen_fold["word_score"])
        gen_sweep.insert(0, "fold", fold_idx)
        gen_sweep["condition"] = "general_sf"
        fold_sweep_records.append(gen_sweep)
    else:
        gen_alpha = gen_beta = None

    # 3) Tune DRA on the same tuning prompts, using this fold's frozen SF beam.
    dra_sweep = tune_dra_on_ids(tune_ids, sf_alpha, sf_beta)
    best_dra_fold = dra_sweep.iloc[0]
    dra_tau = float(best_dra_fold["lambda_tau"])
    dra_psi = float(best_dra_fold["lambda_psi"])
    dra_word = float(best_dra_fold["word_score"])

    dra_sweep.insert(0, "fold", fold_idx)
    dra_sweep["condition"] = "dra"
    fold_sweep_records.append(dra_sweep)

    fold_param_records.append({
        "fold": fold_idx,
        "heldout_sentences": ",".join(sorted(heldout_sents)),
        "sf_lm_weight": sf_alpha, "sf_word_score": sf_beta,
        "general_lm_weight": gen_alpha, "general_word_score": gen_beta,
        "dra_lambda_tau": dra_tau, "dra_lambda_psi": dra_psi,
        "dra_word_score": dra_word, "dra_nbest": DRA_NBEST,
    })

    print(f"  Best Medical SF : alpha={sf_alpha}, beta={sf_beta}, tune WER={best_sf_fold['wer']:.3f}%")
    if decoder_general_only is not None:
        print(f"  Best General SF : alpha={gen_alpha}, beta={gen_beta}, tune WER={best_gen_fold['wer']:.3f}%")
    print(f"  Best DRA        : tau={dra_tau}, psi={dra_psi}, beta={dra_word}, tune WER={best_dra_fold['wer']:.3f}%")

    # Freeze parameters before touching held-out predictions.
    for _, row in test_df.iterrows():
        fid = row["file_id"]
        ref = refs_cache[fid]
        logits = logits_cache[fid]

        hyp_greedy = transcribe_greedy(logits)
        hyp_sf = decode_sf_with_params(logits, sf_alpha, sf_beta, decoder=decoder_sf)
        hyp_dra = decode_dra_with_params(
            logits, dra_tau, dra_psi, dra_word, sf_alpha, sf_beta,
            decoder=decoder_sf, nbest=DRA_NBEST
        )
        hyp_general = (
            decode_sf_with_params(logits, gen_alpha, gen_beta, decoder=decoder_general_only)
            if decoder_general_only is not None else None
        )

        rec = {
            "fold": fold_idx,
            "file_id": fid,
            "speaker_id": row["speaker_id"],
            "state": row["state"],
            "sex": row["sex"],
            "sentence_id": row["sentence_id"],
            "sentence_id_norm": row["sentence_id_norm"],
            "tier": row["tier"],
            "reference": ref,
            "hyp_greedy": hyp_greedy,
            "hyp_sf": hyp_sf,
            "hyp_dra": hyp_dra,
            "hyp_general_sf": hyp_general,
            "wer_greedy": wer(ref, hyp_greedy) * 100,
            "wer_sf": wer(ref, hyp_sf) * 100,
            "wer_dra": wer(ref, hyp_dra) * 100,
        }
        if hyp_general is not None:
            rec["wer_general_sf"] = wer(ref, hyp_general) * 100
        cv_records.append(rec)

df_cv = pd.DataFrame(cv_records).sort_values(["fold", "sentence_id_norm", "speaker_id"]).reset_index(drop=True)
df_fold_params = pd.DataFrame(fold_param_records)
df_fold_sweeps = pd.concat(fold_sweep_records, ignore_index=True)

assert len(df_cv) == 250, f"Expected 250 out-of-fold predictions, got {len(df_cv)}"
assert df_cv["sentence_id_norm"].nunique() == 25
assert df_cv.groupby("sentence_id_norm").size().eq(10).all()
assert df_cv["file_id"].nunique() == 250

print("\nOUT-OF-FOLD EVALUATION COMPLETE")
print(f"  Predictions: {len(df_cv)}")
print(f"  Folds      : {df_cv['fold'].nunique()}")
print(f"  Sentences  : {df_cv['sentence_id_norm'].nunique()}")
print(f"  Speakers   : {df_cv['speaker_id'].nunique()}")



FOLD 1/5
  Tuning prompts : ['001', '002', '003', '004', '005', '006', '007', '008', '009', '011', '012', '013', '014', '015', '018', '019', '022', '023', '024', '025']
  Held-out prompts: ['010', '016', '017', '020', '021']
  Tuning utterances: 200 | Held-out utterances: 50
  Best Medical SF : alpha=0.8, beta=1.5, tune WER=22.115%
  Best General SF : alpha=0.8, beta=1.5, tune WER=23.413%
  Best DRA        : tau=1.0, psi=0.1, beta=0.5, tune WER=21.731%

FOLD 2/5
  Tuning prompts : ['001', '002', '003', '004', '005', '006', '009', '010', '012', '013', '014', '015', '016', '017', '019', '020', '021', '022', '023', '024']
  Held-out prompts: ['007', '008', '011', '018', '025']
  Tuning utterances: 200 | Held-out utterances: 50
  Best Medical SF : alpha=0.8, beta=0.0, tune WER=23.982%
  Best General SF : alpha=0.8, beta=1.5, tune WER=24.615%
  Best DRA        : tau=1.2, psi=0.1, beta=0.5, tune WER=23.937%

FOLD 3/5
  Tuning prompts : ['002', '003', '005', '006', '007', '008', '009', '010'

In [19]:
# Save fold parameters and all tuning sweeps for complete reproducibility.
params_path = f"{RESULTS_DIR}/icasp_fold_parameters.csv"
sweep_path = f"{RESULTS_DIR}/icasp_fold_hyperparameter_sweeps.csv"
cv_path = f"{RESULTS_DIR}/icasp_out_of_fold_predictions.csv"

df_fold_params.to_csv(params_path, index=False, encoding="utf-8")
df_fold_sweeps.to_csv(sweep_path, index=False, encoding="utf-8")
df_cv.to_csv(cv_path, index=False, encoding="utf-8")

print(f"Fold parameters : {params_path}")
print(f"Fold sweeps     : {sweep_path}")
print(f"OOF predictions : {cv_path}")

print("\nFold parameter table:")
display(df_fold_params)

Fold parameters : /content/drive/MyDrive/hausa-asr/6_results/icasp_fold_parameters.csv
Fold sweeps     : /content/drive/MyDrive/hausa-asr/6_results/icasp_fold_hyperparameter_sweeps.csv
OOF predictions : /content/drive/MyDrive/hausa-asr/6_results/icasp_out_of_fold_predictions.csv

Fold parameter table:


,fold,heldout_sentences,sf_lm_weight,sf_word_score,general_lm_weight,general_word_score,dra_lambda_tau,dra_lambda_psi,dra_word_score,dra_nbest
0,1,"010,016,017,020,021",0.8,1.5,0.8,1.5,1.0,0.1,0.5,50
1,2,"007,008,011,018,025",0.8,0.0,0.8,1.5,1.2,0.1,0.5,50
2,3,"001,004,013,019,022",0.8,1.5,0.8,1.5,1.2,0.1,0.5,50
3,4,"003,006,012,015,024",0.8,1.5,0.8,1.5,1.2,0.1,0.5,50
4,5,"002,005,009,014,023",0.8,1.5,0.8,1.5,1.2,0.1,0.5,50


## 13. Cluster-Aware Bootstrap Confidence Intervals and Significance

The final 250 predictions contain 25 repeated sentence prompts × 10 speakers. Therefore, statistical resampling is performed at the **sentence-cluster level**: each bootstrap draw samples 25 sentence IDs with replacement and retains all 10 speaker recordings for every selected sentence. This preserves the dependence induced by repeated prompts.

In [20]:
from jiwer import wer as compute_wer

def _sentence_clusters(df):
    clusters = {}
    for sid, sub in df.groupby("sentence_id_norm", sort=True):
        clusters[sid] = sub.copy()
    return clusters


def clustered_bootstrap_wer_ci(df, hyp_col, n_bootstrap=10000, seed=42):
    """95% percentile CI for micro-WER using sentence-cluster bootstrap."""
    clusters = _sentence_clusters(df)
    sentence_ids = sorted(clusters)
    rng = np.random.default_rng(seed)
    boot_wers = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        sampled = rng.choice(sentence_ids, size=len(sentence_ids), replace=True)
        parts = [clusters[sid] for sid in sampled]
        sample = pd.concat(parts, ignore_index=True)
        boot_wers[b] = compute_wer(
            sample["reference"].tolist(), sample[hyp_col].tolist()
        ) * 100

    lo, hi = np.percentile(boot_wers, [2.5, 97.5])
    return {
        "mean_wer": float(np.mean(boot_wers)),
        "ci_95_lower": float(lo),
        "ci_95_upper": float(hi),
        "bootstrap_values": boot_wers,
    }


def clustered_paired_bootstrap(df, hyp_a, hyp_b, n_bootstrap=10000, seed=42):
    """
    Paired sentence-cluster bootstrap for WER difference A-B.
    Returns a percentile CI and a two-sided null-centered bootstrap p-value.
    """
    clusters = _sentence_clusters(df)
    sentence_ids = sorted(clusters)
    rng = np.random.default_rng(seed)

    observed_a = compute_wer(df["reference"].tolist(), df[hyp_a].tolist()) * 100
    observed_b = compute_wer(df["reference"].tolist(), df[hyp_b].tolist()) * 100
    observed_diff = observed_a - observed_b

    diffs = np.empty(n_bootstrap, dtype=float)
    for b in range(n_bootstrap):
        sampled = rng.choice(sentence_ids, size=len(sentence_ids), replace=True)
        sample = pd.concat([clusters[sid] for sid in sampled], ignore_index=True)
        wa = compute_wer(sample["reference"].tolist(), sample[hyp_a].tolist()) * 100
        wb = compute_wer(sample["reference"].tolist(), sample[hyp_b].tolist()) * 100
        diffs[b] = wa - wb

    lo, hi = np.percentile(diffs, [2.5, 97.5])

    # Null-centered bootstrap: shift the bootstrap distribution so its center
    # is at zero, then ask how often |difference| is at least as extreme as observed.
    null_centered = diffs - np.mean(diffs)
    p_value = float(np.mean(np.abs(null_centered) >= abs(observed_diff)))

    return {
        "observed_diff": float(observed_diff),
        "mean_diff": float(np.mean(diffs)),
        "ci_95_lower": float(lo),
        "ci_95_upper": float(hi),
        "p_value": p_value,
        "significant_at_05": not (lo <= 0 <= hi),
        "bootstrap_values": diffs,
    }


print("Clustered bootstrap: 25 sentence clusters × all 10 speakers")
print("10,000 resamples per statistic")

for name, col in [("Greedy", "hyp_greedy"), ("SF", "hyp_sf"), ("DRA", "hyp_dra")]:
    ci = clustered_bootstrap_wer_ci(df_cv, col, n_bootstrap=10000, seed=42)
    print(f"  {name:<8} WER={ci['mean_wer']:.2f}% "
          f"95% CI=[{ci['ci_95_lower']:.2f}, {ci['ci_95_upper']:.2f}]")

print("\nPaired clustered-bootstrap tests:")
for label, a, b, seed in [
    ("Greedy vs SF", "hyp_greedy", "hyp_sf", 43),
    ("Greedy vs DRA", "hyp_greedy", "hyp_dra", 44),
    ("SF vs DRA", "hyp_sf", "hyp_dra", 45),
]:
    res = clustered_paired_bootstrap(df_cv, a, b, n_bootstrap=10000, seed=seed)
    print(f"  {label:<15} delta={res['observed_diff']:+.2f} "
          f"CI=[{res['ci_95_lower']:+.2f}, {res['ci_95_upper']:+.2f}] "
          f"p={res['p_value']:.4f} "
          f"significant={res['significant_at_05']}")


Clustered bootstrap: 25 sentence clusters × all 10 speakers
10,000 resamples per statistic
  Greedy   WER=29.67% 95% CI=[24.81, 35.24]
  SF       WER=23.67% 95% CI=[19.26, 28.70]
  DRA      WER=23.44% 95% CI=[19.14, 28.32]

Paired clustered-bootstrap tests:
  Greedy vs SF    delta=+6.01 CI=[+4.04, +8.07] p=0.0000 significant=True
  Greedy vs DRA   delta=+6.23 CI=[+4.27, +8.31] p=0.0000 significant=True
  SF vs DRA       delta=+0.22 CI=[-0.25, +0.81] p=0.4230 significant=False


## 14. Micro-Averaged OOF Results Summary

In [21]:
def summarise_micro_cv(df, label):
    refs = df["reference"].tolist()
    out = {"group": label, "n_utterances": len(df)}
    for name, col in [("greedy", "hyp_greedy"), ("sf", "hyp_sf"), ("dra", "hyp_dra")]:
        out[f"wer_{name}"] = compute_wer(refs, df[col].tolist()) * 100
        out[f"cer_{name}"] = compute_cer(refs, df[col].tolist()) * 100
    out["sf_rel_improv"] = (out["wer_greedy"] - out["wer_sf"]) / out["wer_greedy"] * 100
    out["dra_rel_improv"] = (out["wer_greedy"] - out["wer_dra"]) / out["wer_greedy"] * 100
    out["dra_vs_sf_rel"] = (out["wer_sf"] - out["wer_dra"]) / out["wer_sf"] * 100
    return out

summary_rows = []
overall = summarise_micro_cv(df_cv, "OVERALL")
summary_rows.append(overall)

print("OVERALL — pooled out-of-fold evaluation")
print(f"  {'Method':<20} {'WER':>8} {'CER':>8}")
print(f"  {'Greedy':<20} {overall['wer_greedy']:>7.2f}% {overall['cer_greedy']:>7.2f}%")
print(f"  {'Shallow Fusion':<20} {overall['wer_sf']:>7.2f}% {overall['cer_sf']:>7.2f}%")
print(f"  {'Density Ratio':<20} {overall['wer_dra']:>7.2f}% {overall['cer_dra']:>7.2f}%")

print("\nBy sentence tier")
for tier in [1, 2, 3]:
    sub = summarise_micro_cv(df_cv[df_cv["tier"] == tier], f"Tier {tier}")
    summary_rows.append(sub)
    print(f"  Tier {tier}: N={sub['n_utterances']} | "
          f"Greedy={sub['wer_greedy']:.2f}% | SF={sub['wer_sf']:.2f}% | DRA={sub['wer_dra']:.2f}%")

df_summary = pd.DataFrame(summary_rows)


OVERALL — pooled out-of-fold evaluation
  Method                    WER      CER
  Greedy                 29.56%    6.86%
  Shallow Fusion         23.55%    6.07%
  Density Ratio          23.33%    6.08%

By sentence tier
  Tier 1: N=80 | Greedy=41.06% | SF=36.17% | DRA=34.89%
  Tier 2: N=120 | Greedy=27.31% | SF=20.77% | DRA=20.54%
  Tier 3: N=50 | Greedy=26.98% | SF=21.15% | DRA=21.46%


## 15. Paper-Ready Statistical Summary

In [22]:
ci_g = clustered_bootstrap_wer_ci(df_cv, "hyp_greedy", n_bootstrap=10000, seed=42)
ci_s = clustered_bootstrap_wer_ci(df_cv, "hyp_sf", n_bootstrap=10000, seed=42)
ci_d = clustered_bootstrap_wer_ci(df_cv, "hyp_dra", n_bootstrap=10000, seed=42)

gs = clustered_paired_bootstrap(df_cv, "hyp_greedy", "hyp_sf", n_bootstrap=10000, seed=43)
gd = clustered_paired_bootstrap(df_cv, "hyp_greedy", "hyp_dra", n_bootstrap=10000, seed=44)
sd = clustered_paired_bootstrap(df_cv, "hyp_sf", "hyp_dra", n_bootstrap=10000, seed=45)

print(f"Greedy: {overall['wer_greedy']:.2f}% [{ci_g['ci_95_lower']:.2f}, {ci_g['ci_95_upper']:.2f}]")
print(f"SF    : {overall['wer_sf']:.2f}% [{ci_s['ci_95_lower']:.2f}, {ci_s['ci_95_upper']:.2f}]")
print(f"DRA   : {overall['wer_dra']:.2f}% [{ci_d['ci_95_lower']:.2f}, {ci_d['ci_95_upper']:.2f}]")
print(f"\nGreedy vs SF: delta={gs['observed_diff']:+.2f}, CI=[{gs['ci_95_lower']:+.2f},{gs['ci_95_upper']:+.2f}], p={gs['p_value']:.4f}")
print(f"Greedy vs DRA: delta={gd['observed_diff']:+.2f}, CI=[{gd['ci_95_lower']:+.2f},{gd['ci_95_upper']:+.2f}], p={gd['p_value']:.4f}")
print(f"SF vs DRA    : delta={sd['observed_diff']:+.2f}, CI=[{sd['ci_95_lower']:+.2f},{sd['ci_95_upper']:+.2f}], p={sd['p_value']:.4f}")

print("\nRelative WER reductions vs Greedy:")
print(f"  SF : {overall['sf_rel_improv']:.2f}%")
print(f"  DRA: {overall['dra_rel_improv']:.2f}%")


Greedy: 29.56% [24.81, 35.24]
SF    : 23.55% [19.26, 28.70]
DRA   : 23.33% [19.14, 28.32]

Greedy vs SF: delta=+6.01, CI=[+4.04,+8.07], p=0.0000
Greedy vs DRA: delta=+6.23, CI=[+4.27,+8.31], p=0.0000
SF vs DRA    : delta=+0.22, CI=[-0.25,+0.81], p=0.4230

Relative WER reductions vs Greedy:
  SF : 20.32%
  DRA: 21.07%


## 16. Diagnostics — High-WER Out-of-Fold Utterances

In [23]:
print("High-WER utterances (Greedy WER > 80%)")
df_cv["wer_greedy"] = df_cv.apply(lambda r: wer(r["reference"], r["hyp_greedy"]) * 100, axis=1)
df_cv["wer_sf"] = df_cv.apply(lambda r: wer(r["reference"], r["hyp_sf"]) * 100, axis=1)
df_cv["wer_dra"] = df_cv.apply(lambda r: wer(r["reference"], r["hyp_dra"]) * 100, axis=1)

high_wer = df_cv[df_cv["wer_greedy"] > 80].sort_values("wer_greedy", ascending=False)
for _, row in high_wer.iterrows():
    print(f"\nFold {row['fold']} | {row['file_id']} | sentence {row['sentence_id_norm']}")
    print(f"WER: Greedy={row['wer_greedy']:.1f}% SF={row['wer_sf']:.1f}% DRA={row['wer_dra']:.1f}%")
    print(f"Reference: {row['reference']}")
    print(f"Greedy   : {row['hyp_greedy']}")
    print(f"SF       : {row['hyp_sf']}")
    print(f"DRA      : {row['hyp_dra']}")
print(f"\nTotal high-WER utterances: {len(high_wer)}")


High-WER utterances (Greedy WER > 80%)

Fold 2 | S06_GO_M_007 | sentence 007
WER: Greedy=100.0% SF=100.0% DRA=100.0%
Reference: hannuwana na karkarwa
Greedy   : hanuwana ya karkawa
SF       : hanuwana ya karkawa
DRA      : hanuwana ya karkawa

Fold 2 | S20_BA_F_007 | sentence 007
WER: Greedy=100.0% SF=100.0% DRA=100.0%
Reference: hannuwana na karkarwa
Greedy   : hannuwa na na a karkarwa
SF       : hannuwa na na a karkarwa
DRA      : hannuwa na na a karkarwa

Fold 2 | S21_BA_F_007 | sentence 007
WER: Greedy=100.0% SF=66.7% DRA=66.7%
Reference: hannuwana na karkarwa
Greedy   : hannuwa na na kakkarwa
SF       : hannuwa na na karkarwa
DRA      : hannuwa na na karkarwa

Fold 3 | S06_GO_M_004 | sentence 004
WER: Greedy=100.0% SF=100.0% DRA=100.0%
Reference: ina buƙatar magani don ciwon kai
Greedy   : i na-abukata maga naitociwonke
SF       : a na bukata magana itociwonke
DRA      : a na bukatar magana itociwonke

Fold 4 | S06_GO_M_006 | sentence 006
WER: Greedy=100.0% SF=80.0% DRA=80.0%
Refe

## 17. Audio Playback Check

In [ ]:
import IPython.display as ipd

audio_009 = "/content/drive/MyDrive/hausa-asr/5_evaluation_dataset/audio/S08_TA_M/S08_TA_M_009.m4a"
audio_010 = "/content/drive/MyDrive/hausa-asr/5_evaluation_dataset/audio/S08_TA_M/S08_TA_M_010.m4a"

print("Playing S08_TA_M_009")
ipd.display(ipd.Audio(audio_009))
print("Playing S08_TA_M_010")
ipd.display(ipd.Audio(audio_010))

## 18. Targeted Transcription Test

Uses the final **out-of-fold predictions** rather than re-running the acoustic model. This is diagnostic only and is not part of the held-out evaluation protocol.

In [ ]:
TARGET_SENTENCES = {
    "009": "Likita ya ce ina da zazzabin cizon sauro kuma ya rubuta magani",
    "010": "Ka sha kwayar maganin sau uku a kowace rana bayan abinci",
    "011": "An diba jinina a asibiti domin gwaji",
}

for sid, ref in TARGET_SENTENCES.items():
    sub = df_cv[df_cv["sentence_id_norm"] == sid]
    if len(sub) == 0:
        continue
    print(f"\nSentence {sid}: {ref}")
    for _, row in sub.iterrows():
        print(f"  {row['speaker_id']} | Fold {row['fold']} | Greedy={row['wer_greedy']:.1f}% | SF={row['wer_sf']:.1f}% | DRA={row['wer_dra']:.1f}%")
        print(f"    G: {row['hyp_greedy']}")
        print(f"    S: {row['hyp_sf']}")
        print(f"    D: {row['hyp_dra']}")

## 19. Metadata Alignment Check

In [ ]:
print("Metadata alignment check")
print(df_cv.groupby(["fold", "sentence_id_norm"]).size().unstack(fill_value=0))
assert df_cv.groupby("fold").size().eq(50).all()
assert df_cv.groupby("sentence_id_norm").size().eq(10).all()
print("Alignment checks passed: 5 folds × 50 held-out utterances; 25 sentences × 10 speakers.")

## 20. Per-Sentence Analysis

In [24]:
print("Per-sentence pooled across all speakers")
sent_rows = []
for sid in sorted(df_cv["sentence_id_norm"].unique()):
    sub = df_cv[df_cv["sentence_id_norm"] == sid]
    s = summarise_micro_cv(sub, f"Sentence {sid}")
    s["sentence_id"] = sid
    s["tier"] = int(sub["tier"].iloc[0])
    sent_rows.append(s)
    print(f"  {sid} | T{s['tier']} | Greedy={s['wer_greedy']:.2f}% SF={s['wer_sf']:.2f}% DRA={s['wer_dra']:.2f}%")

df_sent_sum = pd.DataFrame(sent_rows)

Per-sentence pooled across all speakers
  001 | T1 | Greedy=34.29% SF=34.29% DRA=31.43%
  002 | T1 | Greedy=34.00% SF=34.00% DRA=34.00%
  003 | T1 | Greedy=53.33% SF=48.33% DRA=41.67%
  004 | T1 | Greedy=25.00% SF=23.33% DRA=23.33%
  005 | T1 | Greedy=55.00% SF=33.75% DRA=33.75%
  006 | T1 | Greedy=30.00% SF=26.00% DRA=26.00%
  007 | T1 | Greedy=76.67% SF=73.33% DRA=73.33%
  008 | T1 | Greedy=32.86% SF=34.29% DRA=34.29%
  009 | T2 | Greedy=9.17% SF=8.33% DRA=8.33%
  010 | T2 | Greedy=28.18% SF=16.36% DRA=16.36%
  011 | T2 | Greedy=35.71% SF=34.29% DRA=34.29%
  012 | T2 | Greedy=22.73% SF=20.00% DRA=19.09%
  013 | T2 | Greedy=12.73% SF=11.82% DRA=12.73%
  014 | T2 | Greedy=21.67% SF=17.50% DRA=17.50%
  015 | T2 | Greedy=16.00% SF=13.00% DRA=14.00%
  016 | T2 | Greedy=33.57% SF=22.14% DRA=22.14%
  017 | T2 | Greedy=51.67% SF=39.17% DRA=40.00%
  018 | T2 | Greedy=19.23% SF=8.46% DRA=5.38%
  019 | T2 | Greedy=38.75% SF=23.75% DRA=23.75%
  020 | T2 | Greedy=46.67% SF=45.56% DRA=44.44%
  021

## 21. Save Core ICASSP Results

In [25]:
utt_path = f"{RESULTS_DIR}/evaluation_results_icasp_oof.csv"
sum_path = f"{RESULTS_DIR}/summary_results_icasp_oof.csv"
sent_path = f"{RESULTS_DIR}/sentence_results_icasp_oof.csv"

df_cv.to_csv(utt_path, index=False, encoding="utf-8")
df_summary.to_csv(sum_path, index=False, encoding="utf-8")
df_sent_sum.to_csv(sent_path, index=False, encoding="utf-8")
print(f"Per-utterance OOF results: {utt_path}")
print(f"Summary results         : {sum_path}")
print(f"Per-sentence results    : {sent_path}")

Per-utterance OOF results: /content/drive/MyDrive/hausa-asr/6_results/evaluation_results_icasp_oof.csv
Summary results         : /content/drive/MyDrive/hausa-asr/6_results/summary_results_icasp_oof.csv
Per-sentence results    : /content/drive/MyDrive/hausa-asr/6_results/sentence_results_icasp_oof.csv


## 22. Paper-Ready Summary

In [26]:
print(f"""
ICASSP out-of-fold evaluation

System            WER (%)    CER (%)    Rel. WER reduction vs Greedy
Greedy (no LM)    {overall['wer_greedy']:>6.2f}     {overall['cer_greedy']:>6.2f}     -
Shallow Fusion    {overall['wer_sf']:>6.2f}     {overall['cer_sf']:>6.2f}     {overall['sf_rel_improv']:>+.1f}%
Density Ratio     {overall['wer_dra']:>6.2f}     {overall['cer_dra']:>6.2f}     {overall['dra_rel_improv']:>+.1f}%

25 sentence prompts × 10 speakers = {len(df_cv)} out-of-fold utterances
5 folds; 20 prompts tune / 5 prompts held out per fold
Cluster bootstrap: 25 sentence clusters, 10,000 resamples

Greedy vs SF p = {gs['p_value']:.4f}
Greedy vs DRA p = {gd['p_value']:.4f}
SF vs DRA     p = {sd['p_value']:.4f}
""")


ICASSP out-of-fold evaluation

System            WER (%)    CER (%)    Rel. WER reduction vs Greedy
Greedy (no LM)     29.56       6.86     -
Shallow Fusion     23.55       6.07     +20.3%
Density Ratio      23.33       6.08     +21.1%

25 sentence prompts × 10 speakers = 250 out-of-fold utterances
5 folds; 20 prompts tune / 5 prompts held out per fold
Cluster bootstrap: 25 sentence clusters, 10,000 resamples

Greedy vs SF p = 0.0000
Greedy vs DRA p = 0.0000
SF vs DRA     p = 0.4230



## 23. SF-Beam → DRA Rescoring — Cross-Validated Methodology

In [27]:
changed = (df_cv["hyp_sf"] != df_cv["hyp_dra"]).sum()
pct_changed = changed / len(df_cv) * 100
print(f"DRA n-best rescoring: n={DRA_NBEST}, beam width={BEAM_WIDTH}")
print("For every fold, DRA candidates come from the fold-specific SF beam, whose alpha/beta were selected using only the fold's 20 tuning prompts.")
print(f"Rescoring changed the SF hypothesis in {changed}/{len(df_cv)} OOF utterances ({pct_changed:.1f}%).")


DRA n-best rescoring: n=50, beam width=100
For every fold, DRA candidates come from the fold-specific SF beam, whose alpha/beta were selected using only the fold's 20 tuning prompts.
Rescoring changed the SF hypothesis in 28/250 OOF utterances (11.2%).


## 24. Tier / Sex / Speaker Breakdown

In [28]:
print("Tier breakdown")
for tier in [1, 2, 3]:
    sub = summarise_micro_cv(df_cv[df_cv["tier"] == tier], f"Tier {tier}")
    rel = sub["dra_rel_improv"]
    print(f"  Tier {tier}: N={sub['n_utterances']} | Greedy={sub['wer_greedy']:.2f}% | SF={sub['wer_sf']:.2f}% | DRA={sub['wer_dra']:.2f}% | DRA rel={rel:+.1f}%")

print("\nSex breakdown")
for sex, label in [("M", "Male"), ("F", "Female")]:
    subdf = df_cv[df_cv["sex"] == sex]
    if len(subdf):
        s = summarise_micro_cv(subdf, label)
        print(f"  {label}: N={s['n_utterances']} | Greedy={s['wer_greedy']:.2f}% | SF={s['wer_sf']:.2f}% | DRA={s['wer_dra']:.2f}%")

print("\nSpeaker breakdown")
for spk in sorted(df_cv["speaker_id"].unique()):
    subdf = df_cv[df_cv["speaker_id"] == spk]
    s = summarise_micro_cv(subdf, spk)
    print(f"  {spk}: Greedy={s['wer_greedy']:.2f}% SF={s['wer_sf']:.2f}% DRA={s['wer_dra']:.2f}%")

Tier breakdown
  Tier 1: N=80 | Greedy=41.06% | SF=36.17% | DRA=34.89% | DRA rel=+15.0%
  Tier 2: N=120 | Greedy=27.31% | SF=20.77% | DRA=20.54% | DRA rel=+24.8%
  Tier 3: N=50 | Greedy=26.98% | SF=21.15% | DRA=21.46% | DRA rel=+20.5%

Sex breakdown
  Male: N=125 | Greedy=28.86% | SF=23.08% | DRA=23.00%
  Female: N=125 | Greedy=30.26% | SF=24.03% | DRA=23.66%

Speaker breakdown
  S04: Greedy=26.01% SF=22.34% DRA=22.71%
  S06: Greedy=34.80% SF=27.84% DRA=28.21%
  S08: Greedy=32.60% SF=25.64% DRA=24.91%
  S10: Greedy=26.37% SF=20.88% DRA=20.51%
  S11: Greedy=24.54% SF=18.68% DRA=18.68%
  S13: Greedy=25.64% SF=21.61% DRA=20.15%
  S18: Greedy=27.11% SF=22.71% DRA=21.98%
  S20: Greedy=29.30% SF=22.34% DRA=22.34%
  S21: Greedy=31.50% SF=23.08% DRA=23.44%
  S23: Greedy=37.73% SF=30.40% DRA=30.40%


## 25. Domain-Specific Medical Lexicon (Corpus Frequency Contrast)

In [29]:
import kenlm

print("Building medical lexicon from corpus frequency contrast")
lm_medical = kenlm.Model(LM_TGT)
lm_general = kenlm.Model(LM_SRC)

all_ref_words = set()
for _, row in df_cv.iterrows():
    all_ref_words.update(row["reference"].lower().split())

word_scores = {}
for word in all_ref_words:
    if len(word) < 2:
        continue
    lp_med = lm_medical.score(word, bos=False, eos=False)
    lp_gen = lm_general.score(word, bos=False, eos=False)
    word_scores[word] = {"lp_medical": lp_med, "lp_general": lp_gen, "ratio": lp_med - lp_gen}

MEDICAL_THRESHOLD = 0.5
medical_lexicon = {w:s for w,s in word_scores.items() if s["ratio"] > MEDICAL_THRESHOLD}
general_lexicon = {w:s for w,s in word_scores.items() if s["ratio"] < -MEDICAL_THRESHOLD}
neutral_words = {w:s for w,s in word_scores.items() if abs(s["ratio"]) <= MEDICAL_THRESHOLD}

print(f"Medical words : {len(medical_lexicon)}")
print(f"General words : {len(general_lexicon)}")
print(f"Neutral words : {len(neutral_words)}")

lexicon_path = f"{RESULTS_DIR}/medical_lexicon_derived_icasp.json"
with open(lexicon_path, "w", encoding="utf-8") as f:
    json.dump({"medical_terms": list(medical_lexicon.keys()), "general_terms": list(general_lexicon.keys()), "neutral_terms": list(neutral_words.keys()), "threshold": MEDICAL_THRESHOLD, "word_scores": word_scores}, f, ensure_ascii=False, indent=2)
print(f"Saved: {lexicon_path}")

Building medical lexicon from corpus frequency contrast
Medical words : 48
General words : 4
Neutral words : 101
Saved: /content/drive/MyDrive/hausa-asr/6_results/medical_lexicon_derived_icasp.json


## 26. Medical Term Error Rate (MTER)

MTER is computed on the final out-of-fold hypotheses using the same fixed 64-term lexicon and occurrence-based, position-independent matching used in the workshop analysis. It is recall-oriented and should be interpreted alongside WER/CER.

In [30]:
# Fixed lexicon carried forward from the submitted experiment.
CORPUS_DERIVED_TERMS = {
    "tiyata", "zazzabi", "ƙwayar", "kwayar", "ƙashin", "kashin",
    "sukari", "gwaji", "karkarwa", "sauro", "allurar", "cizon",
    "jini", "ciwon", "ciwo", "zazzabin", "makonni", "likita",
    "gwajin", "buƙaci", "bukaci", "cutar", "kise", "jikinna",
    "rigakafi", "asibiti", "magani", "maganin", "magunguna",
    "matakan", "matsin", "jinina", "kulawa", "iyaka", "alamu",
    "rashin", "rawar", "gudawa", "rauni", "amai", "tari",
    "tsanani", "musamman", "zuciyata", "haɗari", "hadari",
    "buƙatar", "bukatar",
}
EXPERT_VALIDATED_TERMS = {
    "diba", "gwaji", "gwajin", "asibiti", "duba", "zafi", "ƙishirwa", "kishirwa", "gudawa", "rawar",
    "karkarwa", "rashin", "lafiya", "hawan", "magani", "maganin", "magunguna", "ƙwayar", "kwayar",
    "allurar", "rigakafi", "tiyata", "kulawa", "jini", "jinina", "jiki", "jikina", "jikansa", "jikinna",
    "kirjina", "kirji", "ƙashin", "kashin", "baya", "hannuwana", "zazzabi", "zazzabin", "cizon", "sauro",
    "sukari", "cutar", "ciwon", "ciwo", "tari", "amai", "tsanani", "musamman", "iyaka", "matakan", "matsin",
    "alamu", "haɗari", "hadari", "makonni", "sha", "warke", "rubuta", "dawo",
}
CLINICAL_CATEGORIES = {
    "Symptoms & Conditions": {"zazzabi","zazzabin","cizon","sauro","sukari","cutar","ciwon","ciwo","tari","amai","gudawa","rawar","rauni","karkarwa","rashin","zafi","ƙishirwa","kishirwa","hawan","tsanani","haɗari","hadari"},
    "Medications & Treatments": {"magani","maganin","magunguna","ƙwayar","kwayar","allurar","rigakafi","tiyata","kulawa","kise","sha","warke"},
    "Clinical Procedures": {"gwaji","gwajin","diba","duba","rubuta","dawo"},
    "Medical Personnel & Settings": {"likita","asibiti"},
    "Anatomy": {"jini","jinina","jiki","jikina","jikansa","jikinna","kirjina","kirji","ƙashin","kashin","baya","hannuwana"},
    "Clinical Descriptors": {"musamman","iyaka","matakan","matsin","alamu","makonni","tsanani","buƙaci","bukaci","buƙatar","bukatar","zuciyata"},
}
ALL_MEDICAL_TERMS = CORPUS_DERIVED_TERMS.union(EXPERT_VALIDATED_TERMS)

def normalize_hausa(word):
    return word.replace("ƙ", "k").replace("ɗ", "d").replace("ɓ", "b")

ALL_MEDICAL_TERMS_NORM = {normalize_hausa(t) for t in ALL_MEDICAL_TERMS}

def extract_medical_terms(text):
    return [w for w in text.lower().split() if w in ALL_MEDICAL_TERMS or normalize_hausa(w) in ALL_MEDICAL_TERMS_NORM]

def compute_mter_utterance(reference, hypothesis):
    ref_terms = extract_medical_terms(reference)
    if not ref_terms:
        return {"total_terms": 0, "correct_terms": 0, "mter": None}
    hyp_words = set(hypothesis.lower().split())
    hyp_norm = {normalize_hausa(w) for w in hyp_words}
    correct = sum(1 for term in ref_terms if term in hyp_words or normalize_hausa(term) in hyp_norm)
    return {"total_terms": len(ref_terms), "correct_terms": correct, "mter": (1 - correct / len(ref_terms)) * 100}

mter_rows = []
for _, row in df_cv.iterrows():
    g = compute_mter_utterance(row["reference"], row["hyp_greedy"])
    s = compute_mter_utterance(row["reference"], row["hyp_sf"])
    d = compute_mter_utterance(row["reference"], row["hyp_dra"])
    if g["total_terms"] == 0:
        continue
    mter_rows.append({
        "file_id": row["file_id"], "fold": row["fold"], "speaker_id": row["speaker_id"],
        "state": row["state"], "sex": row["sex"], "tier": row["tier"], "sentence_id": row["sentence_id"],
        "reference": row["reference"], "total_terms": g["total_terms"],
        "correct_greedy": g["correct_terms"], "correct_sf": s["correct_terms"], "correct_dra": d["correct_terms"],
        "mter_greedy": g["mter"], "mter_sf": s["mter"], "mter_dra": d["mter"],
    })

df_mter = pd.DataFrame(mter_rows)
total_terms = int(df_mter["total_terms"].sum())
correct_g = int(df_mter["correct_greedy"].sum())
correct_sf = int(df_mter["correct_sf"].sum())
correct_dra = int(df_mter["correct_dra"].sum())
mter_g = (1 - correct_g / total_terms) * 100
mter_sf = (1 - correct_sf / total_terms) * 100
mter_dra = (1 - correct_dra / total_terms) * 100

print(f"MTER terms: {total_terms}")
print(f"Greedy: {mter_g:.2f}%")
print(f"SF    : {mter_sf:.2f}%")
print(f"DRA   : {mter_dra:.2f}%")
print(f"SF relative MTER reduction : {(mter_g-mter_sf)/mter_g*100:.1f}%")
print(f"DRA relative MTER reduction: {(mter_g-mter_dra)/mter_g*100:.1f}%")

# Category-level pooled MTER.
def category_counts(df, cat_terms):
    cat_norm = {normalize_hausa(t) for t in cat_terms}
    total = g = s = d = 0
    for _, row in df.iterrows():
        refs = [t for t in extract_medical_terms(row["reference"]) if t in cat_terms or normalize_hausa(t) in cat_norm]
        hg = {normalize_hausa(w) for w in row["hyp_greedy"].lower().split()}
        hs = {normalize_hausa(w) for w in row["hyp_sf"].lower().split()}
        hd = {normalize_hausa(w) for w in row["hyp_dra"].lower().split()}
        for term in refs:
            tn = normalize_hausa(term); total += 1
            g += tn in hg; s += tn in hs; d += tn in hd
    return total, g, s, d

category_results = []
for cat, terms in CLINICAL_CATEGORIES.items():
    total, g, s, d = category_counts(df_cv, terms)
    if total == 0: continue
    mg, ms, md = (1-g/total)*100, (1-s/total)*100, (1-d/total)*100
    category_results.append({"category":cat,"total":total,"mter_greedy":mg,"mter_sf":ms,"mter_dra":md,"dra_rel":(mg-md)/mg*100 if mg else 0})
    print(f"{cat:<30} N={total:>4} Greedy={mg:>6.2f}% SF={ms:>6.2f}% DRA={md:>6.2f}%")

# Save MTER outputs.
df_mter.to_csv(f"{RESULTS_DIR}/mter_results_icasp.csv", index=False, encoding="utf-8")
pd.DataFrame(category_results).to_csv(f"{RESULTS_DIR}/mter_by_category_icasp.csv", index=False, encoding="utf-8")


MTER terms: 790
Greedy: 31.27%
SF    : 24.18%
DRA   : 24.18%
SF relative MTER reduction : 22.7%
DRA relative MTER reduction: 22.7%
Symptoms & Conditions          N= 260 Greedy= 19.62% SF= 14.62% DRA= 14.23%
Medications & Treatments       N= 150 Greedy= 48.00% SF= 30.67% DRA= 30.67%
Clinical Procedures            N=  70 Greedy= 24.29% SF= 14.29% DRA= 14.29%
Medical Personnel & Settings   N=  60 Greedy=  6.67% SF=  1.67% DRA=  1.67%
Anatomy                        N= 120 Greedy= 56.67% SF= 57.50% DRA= 59.17%
Clinical Descriptors           N= 140 Greedy= 25.71% SF= 20.00% DRA= 19.29%


## 27. Properly Tuned General-LM-Only SF Ablation

This ablation uses the **same sentence-level folds** as the main experiment. General-LM SF parameters are selected on the 20 tuning prompts of each fold and frozen before decoding the five held-out prompts. This prevents an unfair comparison caused by hard-coded general-LM parameters.

In [31]:
if "hyp_general_sf" in df_cv.columns and df_cv["hyp_general_sf"].notna().all():
    refs = df_cv["reference"].tolist()
    gen_hyps = df_cv["hyp_general_sf"].tolist()
    overall_wer_general_sf = compute_wer(refs, gen_hyps) * 100

    gen_mter_rows = []
    for _, row in df_cv.iterrows():
        r = compute_mter_utterance(row["reference"], row["hyp_general_sf"])
        if r["total_terms"]:
            gen_mter_rows.append(r)
    gen_total = sum(r["total_terms"] for r in gen_mter_rows)
    gen_correct = sum(r["correct_terms"] for r in gen_mter_rows)
    mter_general_sf = (1 - gen_correct / gen_total) * 100 if gen_total else np.nan

    print("General-LM-only SF (properly fold-tuned)")
    print(f"  WER  = {overall_wer_general_sf:.2f}%")
    print(f"  MTER = {mter_general_sf:.2f}%")
else:
    overall_wer_general_sf = np.nan
    mter_general_sf = np.nan
    print("General LM not available; ablation skipped.")


General-LM-only SF (properly fold-tuned)
  WER  = 23.96%
  MTER = 23.80%


## 28. SF + Combined (General + Medical) LM

This is an optional ablation retained from the workshop analysis. The combined LM is trained once from the same general and medical corpora; its decoding weights are tuned separately within each sentence-level fold before held-out evaluation.

In [36]:
print("Preparing combined general + medical 4-gram LM")

# ------------------------------------------------------------------
# KenLM setup
# ------------------------------------------------------------------
KENLM_BUILD_DIR = "/content/kenlm_build"
KENLM_BIN_DIR = f"{KENLM_BUILD_DIR}/build/bin"

# Make sure an existing KenLM installation is visible.
if os.path.exists(KENLM_BIN_DIR):
    os.environ["PATH"] = f"{KENLM_BIN_DIR}:{os.environ.get('PATH', '')}"

# Only compile KenLM if the command-line tools are unavailable.
if shutil.which("lmplz") is None or shutil.which("build_binary") is None:
    print("  KenLM tools not found. Installing/building KenLM...")

    apt_update = subprocess.run(
        "apt-get update -qq",
        shell=True,
        capture_output=True,
        text=True
    )

    if apt_update.returncode != 0:
        print("  apt-get update warning:")
        print(apt_update.stderr[-2000:])

    apt_install = subprocess.run(
        "DEBIAN_FRONTEND=noninteractive "
        "apt-get install -y -qq "
        "build-essential cmake "
        "libboost-all-dev "
        "libbz2-dev liblzma-dev zlib1g-dev",
        shell=True,
        capture_output=True,
        text=True
    )

    if apt_install.returncode != 0:
        print("  apt-get installation failed:")
        print(apt_install.stderr[-3000:])
        raise RuntimeError(
            "Could not install KenLM build dependencies. "
            "Restart the runtime and rerun the setup cells."
        )

    subprocess.run(
        f"rm -rf {KENLM_BUILD_DIR} && mkdir -p {KENLM_BUILD_DIR}",
        shell=True,
        check=True
    )

    download_result = subprocess.run(
        f"wget -q -O - https://kheafield.com/code/kenlm.tar.gz "
        f"| tar xz -C {KENLM_BUILD_DIR} --strip-components=1",
        shell=True,
        capture_output=True,
        text=True
    )

    if download_result.returncode != 0:
        raise RuntimeError(
            "Failed to download KenLM source:\n"
            + download_result.stderr[-3000:]
        )

    build_dir = f"{KENLM_BUILD_DIR}/build"
    os.makedirs(build_dir, exist_ok=True)

    build_result = subprocess.run(
        f"cd {build_dir} && "
        f"cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && "
        f"make -j2 lmplz build_binary",
        shell=True,
        capture_output=True,
        text=True
    )

    if build_result.returncode != 0:
        raise RuntimeError(
            "KenLM compilation failed:\n"
            + build_result.stderr[-5000:]
        )

    os.environ["PATH"] = (
        f"{KENLM_BIN_DIR}:{os.environ.get('PATH', '')}"
    )

lmplz_path = shutil.which("lmplz")
build_binary_path = shutil.which("build_binary")

if lmplz_path is None or build_binary_path is None:
    raise RuntimeError(
        f"KenLM tools unavailable. "
        f"lmplz={lmplz_path}, build_binary={build_binary_path}"
    )

print(f"  lmplz: {lmplz_path}")
print(f"  build_binary: {build_binary_path}")


# ------------------------------------------------------------------
# Load general corpus
# ------------------------------------------------------------------
GENERAL_CORPUS_PATH = f"{BASE_DIR}/4_data/hausa_general_corpus.txt"

with open(GENERAL_CORPUS_PATH, encoding="utf-8") as f:
    general_corpus_lines = [
        line.strip()
        for line in f
        if line.strip()
    ]


# ------------------------------------------------------------------
# Clean corpus lines for KenLM
# ------------------------------------------------------------------
def clean_kenlm_corpus_line(line):
    """
    Remove KenLM special symbols that cannot appear as ordinary
    words in an lmplz training corpus.
    """
    line = str(line).strip()

    line = line.replace("<s>", "")
    line = line.replace("</s>", "")
    line = line.replace("<unk>", "")

    # Normalize whitespace after removing special tokens.
    line = " ".join(line.split())

    return line


medical_clean_lines = [
    clean_kenlm_corpus_line(line)
    for line in corpus_lines
]

general_clean_lines = [
    clean_kenlm_corpus_line(line)
    for line in general_corpus_lines
]

medical_clean_lines = [
    line for line in medical_clean_lines
    if line
]

general_clean_lines = [
    line for line in general_clean_lines
    if line
]

print(f"  General corpus sentences: {len(general_clean_lines):,}")
print(f"  Medical corpus sentences: {len(medical_clean_lines):,}")


# ------------------------------------------------------------------
# Build combined corpus
# ------------------------------------------------------------------
combined_corpus_path = (
    f"{RESULTS_DIR}/hausa_combined_corpus_icasp.txt"
)

with open(combined_corpus_path, "w", encoding="utf-8") as f:

    for line in medical_clean_lines:
        f.write(line + "\n")

    for line in general_clean_lines:
        f.write(line + "\n")

combined_sentence_count = (
    len(medical_clean_lines)
    + len(general_clean_lines)
)

print(
    f"  Combined corpus sentences: "
    f"{combined_sentence_count:,}"
)

print(
    f"  Combined corpus written to: "
    f"{combined_corpus_path}"
)


# ------------------------------------------------------------------
# Train combined 4-gram KenLM
# ------------------------------------------------------------------
combined_arpa_path = (
    f"{RESULTS_DIR}/hausa_combined_4gram_icasp.arpa"
)

combined_lm_path = (
    f"{RESULTS_DIR}/hausa_combined_4gram_icasp.bin"
)

combined_log_path = (
    f"{RESULTS_DIR}/combined_lmplz_icasp.log"
)

# Remove incomplete artifacts from any previous failed run.
for path in [
    combined_arpa_path,
    combined_lm_path
]:
    if os.path.exists(path):
        os.remove(path)

print("  Training combined 4-gram LM...")

result = subprocess.run(
    f"lmplz -o 4 --discount_fallback "
    f"< {combined_corpus_path} "
    f"> {combined_arpa_path} "
    f"2> {combined_log_path}",
    shell=True
)

if result.returncode != 0:
    raise RuntimeError(
        "KenLM lmplz failed. "
        f"Inspect: {combined_log_path}"
    )

print("  Converting ARPA model to binary...")

subprocess.run(
    f"build_binary "
    f"{combined_arpa_path} "
    f"{combined_lm_path}",
    shell=True,
    check=True
)

print(f"Combined LM: {combined_lm_path}")


# ------------------------------------------------------------------
# Separate combined-LM decoder
# ------------------------------------------------------------------
decoder_combined = build_ctcdecoder(
    labels=vocab_list,
    kenlm_model_path=combined_lm_path,
    alpha=0.5,
    beta=1.0
)


# ------------------------------------------------------------------
# Tune combined-LM shallow fusion independently per fold
# ------------------------------------------------------------------
combined_oof = []
combined_params = []

for fold_idx, heldout_sents in enumerate(
    fold_sentence_ids,
    start=1
):

    heldout_sents = set(heldout_sents)

    tune_ids = df_eval[
        ~df_eval["sentence_id_norm"].isin(heldout_sents)
    ]["file_id"].tolist()

    test_df = df_eval[
        df_eval["sentence_id_norm"].isin(heldout_sents)
    ]

    refs = [
        refs_cache[fid]
        for fid in tune_ids
    ]

    rows = []

    for alpha in LM_WEIGHT_GRID:

        for beta in WORD_SCORE_GRID:

            set_sf_lm_params(
                decoder_combined,
                alpha,
                beta
            )

            hyps = [
                decoder_combined.decode(
                    logits_cache[fid],
                    beam_width=BEAM_WIDTH,
                    beam_prune_logp=-10.0,
                    token_min_logp=-5.0
                )
                for fid in tune_ids
            ]

            rows.append({
                "alpha": alpha,
                "beta": beta,
                "wer": compute_wer(
                    refs,
                    hyps
                ) * 100
            })

    sweep_df = (
        pd.DataFrame(rows)
        .sort_values(
            ["wer", "alpha", "beta"]
        )
        .reset_index(drop=True)
    )

    best = sweep_df.iloc[0]

    alpha = float(best["alpha"])
    beta = float(best["beta"])

    combined_params.append({
        "fold": fold_idx,
        "alpha": alpha,
        "beta": beta,
        "tune_wer": float(best["wer"])
    })

    # Freeze selected parameters before held-out evaluation.
    set_sf_lm_params(
        decoder_combined,
        alpha,
        beta
    )

    for _, row in test_df.iterrows():

        fid = row["file_id"]
        ref = refs_cache[fid]

        hyp = decoder_combined.decode(
            logits_cache[fid],
            beam_width=BEAM_WIDTH,
            beam_prune_logp=-10.0,
            token_min_logp=-5.0
        )

        combined_oof.append({
            "fold": fold_idx,
            "file_id": fid,
            "reference": ref,
            "hyp_combined_sf": hyp
        })


# ------------------------------------------------------------------
# Combined-LM OOF results
# ------------------------------------------------------------------
df_combined_oof = pd.DataFrame(
    combined_oof
)

assert len(df_combined_oof) == 250, (
    "Expected 250 combined-LM OOF predictions, "
    f"got {len(df_combined_oof)}"
)

wer_combined_sf = (
    compute_wer(
        df_combined_oof["reference"].tolist(),
        df_combined_oof["hyp_combined_sf"].tolist()
    ) * 100
)


# ------------------------------------------------------------------
# Combined-LM MTER
# ------------------------------------------------------------------
combined_mter = []

for _, r in df_combined_oof.iterrows():

    x = compute_mter_utterance(
        r["reference"],
        r["hyp_combined_sf"]
    )

    if x["total_terms"]:
        combined_mter.append(x)

ct = sum(
    x["total_terms"]
    for x in combined_mter
)

cc = sum(
    x["correct_terms"]
    for x in combined_mter
)

mter_combined_sf = (
    (1 - cc / ct) * 100
    if ct
    else np.nan
)


# ------------------------------------------------------------------
# Save results
# ------------------------------------------------------------------
print()
print(
    f"Combined-LM SF OOF WER : "
    f"{wer_combined_sf:.2f}%"
)

print(
    f"Combined-LM SF OOF MTER: "
    f"{mter_combined_sf:.2f}%"
)

pd.DataFrame(
    combined_params
).to_csv(
    f"{RESULTS_DIR}/combined_lm_fold_parameters_icasp.csv",
    index=False
)

df_combined_oof.to_csv(
    f"{RESULTS_DIR}/sf_combined_lm_results_icasp.csv",
    index=False
)

print("Combined-LM results saved.")

Preparing combined general + medical 4-gram LM
  lmplz: /content/kenlm_build/build/bin/lmplz
  build_binary: /content/kenlm_build/build/bin/build_binary
  General corpus sentences: 75,372
  Medical corpus sentences: 19,420
  Combined corpus sentences: 94,792
  Combined corpus written to: /content/drive/MyDrive/hausa-asr/6_results/hausa_combined_corpus_icasp.txt
  Training combined 4-gram LM...
  Converting ARPA model to binary...


Combined LM: /content/drive/MyDrive/hausa-asr/6_results/hausa_combined_4gram_icasp.bin

Combined-LM SF OOF WER : 23.85%
Combined-LM SF OOF MTER: 23.67%
Combined-LM results saved.


In [35]:
combined_log_path = f"{RESULTS_DIR}/combined_lmplz_icasp.log"

print("===== KenLM lmplz log =====")

if os.path.exists(combined_log_path):
    with open(combined_log_path, encoding="utf-8", errors="replace") as f:
        log = f.read()

    print(log[-10000:])
else:
    print("Log file was not created.")

===== KenLM lmplz log =====
=== 1/5 Counting and sorting n-grams ===
Reading /content/drive/MyDrive/hausa-asr/6_results/hausa_combined_corpus_icasp.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
/content/kenlm_build/lm/builder/corpus_count.cc:179 in void lm::builder::{anonymous}::ComplainDisallowed(StringPiece, lm::WarningAction&) threw FormatLoadException.
Special word <s> is not allowed in the corpus.  I plan to support models containing <unk> in the future.  Pass --skip_symbols to convert these symbols to whitespace.
Aborted (core dumped)



## 29. N-gram Overlap Analysis

The overlap analysis is kept as a contamination/memorization diagnostic. It is **not** treated as evidence of zero contamination; the paper should report the measured exact 4-gram overlap honestly.

In [37]:
# Reuse the medical corpus loaded in Section 5.
N_GRAM_N=4
corpus_ngrams=set()
for line in corpus_lines:
    corpus_ngrams |= get_ngrams(line,N_GRAM_N)

overlap_rows=[]
for sid,sub in df_eval.groupby("sentence_id_norm"):
    text=str(sub["hausa_reference"].iloc[0])
    grams=get_ngrams(text,N_GRAM_N); matched=grams & corpus_ngrams
    overlap_rows.append({"sentence_id":sid,"n_ngrams":len(grams),"n_matched":len(matched)})
df_overlap=pd.DataFrame(overlap_rows)

print(f"Unique training {N_GRAM_N}-grams: {len(corpus_ngrams):,}")
print(f"Evaluation sentences with >=1 overlap: {(df_overlap['n_matched']>0).sum()}/{len(df_overlap)}")
print(f"Pooled matched {N_GRAM_N}-grams: {df_overlap['n_matched'].sum()}/{df_overlap['n_ngrams'].sum()} = {df_overlap['n_matched'].sum()/df_overlap['n_ngrams'].sum()*100:.1f}%")
df_overlap.to_csv(f"{RESULTS_DIR}/ngram_overlap_analysis_icasp.csv",index=False)

Unique training 4-grams: 422,112
Evaluation sentences with >=1 overlap: 9/25
Pooled matched 4-grams: 18/198 = 9.1%


## 30. Domain Contrast Significance Test

In [38]:
from scipy.stats import mannwhitneyu

GENERAL_CORPUS_PATH = f"{BASE_DIR}/4_data/hausa_general_corpus.txt"
with open(GENERAL_CORPUS_PATH, encoding="utf-8") as f:
    general_corpus_lines = [line.strip() for line in f if line.strip()]
held_out = general_corpus_lines[-500:]
medical_sentences = df_eval.groupby("sentence_id_norm")["hausa_reference"].first().tolist()
general_sentences = held_out[:len(medical_sentences)]

def compute_ratio(sentence,lm_med,lm_gen):
    text=normalize_text(sentence)
    return lm_med.score(text,bos=True,eos=True)-lm_gen.score(text,bos=True,eos=True)

medical_ratios=[compute_ratio(s,lm_medical,lm_general) for s in medical_sentences]
general_ratios=[compute_ratio(s,lm_medical,lm_general) for s in general_sentences]
u_stat,p_diff=mannwhitneyu(medical_ratios,general_ratios,alternative="greater")
print(f"Medical mean ratio: {np.mean(medical_ratios):.2f}")
print(f"General mean ratio: {np.mean(general_ratios):.2f}")
print(f"Mann-Whitney U p-value: {p_diff:.6g}")

Medical mean ratio: 3.27
General mean ratio: -18.04
Mann-Whitney U p-value: 4.14026e-09


## 31. Final Reproducibility Manifest

In [39]:
manifest = {
    "protocol": "sentence-level 5-fold CV; 20 prompts tune / 5 prompts held out; all 10 speakers retained",
    "n_folds": N_FOLDS,
    "n_sentence_prompts": len(unique_sentences),
    "n_speakers": int(df_cv["speaker_id"].nunique()),
    "n_oof_utterances": len(df_cv),
    "beam_width": BEAM_WIDTH,
    "dra_nbest": DRA_NBEST,
    "bootstrap_resamples": 10000,
    "bootstrap_cluster": "sentence_id_norm",
    "acoustic_model": MODEL_NAME,
    "sf_grid": {"lm_weight": LM_WEIGHT_GRID, "word_score": WORD_SCORE_GRID},
    "dra_grid": {"lambda_tau": LAMBDA_TAU_GRID, "lambda_psi": LAMBDA_PSI_GRID, "word_score": DRA_WORD_SCORE_GRID},
}

with open(f"{RESULTS_DIR}/icasp_reproducibility_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2,ensure_ascii=False)

print(json.dumps(manifest,indent=2))
print("\nFINAL CHECKS PASSED")
assert len(df_cv)==250
assert df_cv.groupby("fold").size().eq(50).all()
assert df_cv.groupby("sentence_id_norm").size().eq(10).all()
assert df_fold_params.shape[0]==5
assert set(df_cv["sentence_id_norm"])==set(unique_sentences)


{
  "protocol": "sentence-level 5-fold CV; 20 prompts tune / 5 prompts held out; all 10 speakers retained",
  "n_folds": 5,
  "n_sentence_prompts": 25,
  "n_speakers": 10,
  "n_oof_utterances": 250,
  "beam_width": 100,
  "dra_nbest": 50,
  "bootstrap_resamples": 10000,
  "bootstrap_cluster": "sentence_id_norm",
  "acoustic_model": "facebook/mms-1b-fl102",
  "sf_grid": {
    "lm_weight": [
      0.2,
      0.35,
      0.5,
      0.65,
      0.8,
      1.0
    ],
    "word_score": [
      -1.0,
      -0.5,
      0.0,
      0.5,
      1.0,
      1.5
    ]
  },
  "dra_grid": {
    "lambda_tau": [
      0.3,
      0.5,
      0.8,
      1.0,
      1.2,
      1.5
    ],
    "lambda_psi": [
      0.1,
      0.3,
      0.5,
      0.8,
      1.0
    ],
    "word_score": [
      0.5,
      1.0,
      1.5
    ]
  }
}

FINAL CHECKS PASSED
